In [ ]:
import os
import requests
import json
import time
import re
import random
import unicodedata
from collections import defaultdict

# function to normalize text for parsing
def normalize_text(text):
    # Normalize unicode characters, remove markdown formatting, lowercase, and strip dashes
    if not text:
        return text
    
    text = str(text)
    text = text.replace('*', '').replace('<<', '').replace('>>', '')
    text = re.sub(r'[-\u2010-\u2015\u2212\uFE58\uFE63\uFF0D]', '', text)
    normalized = unicodedata.normalize('NFKD', text)
    ascii_text = normalized.encode('ASCII', 'ignore').decode('ASCII')

    # Clean up whitespace + lowercase
    return ' '.join(ascii_text.split()).lower()

# dummy data
user_row = {
    "id": 1,
    "name": "John Doe",
    "username": "johndoe123",
    "description": "Tech enthusiast and blogger.",
    "location": "New York, USA",
    "public_metrics.followers_count": 250,
    "tweets_struct": "Just posted a new blog about AI!",
    "profile_image_main_info": "A smiling man with glasses"
}

belief_features = [
    {
      "title": "ETHNICITY",
      "options": {
        "E1": "white - individuals with origins in any of the original peoples of europe, including, for example, english, german, irish, italian, polish, and scottish -- as well as arab or middle-eastern with origins in any of the original peoples of the middle east or north africa, including, for example, lebanese, iranian, egyptian, syrian, iraqi, and israeli.",
        "E2": "black or african american - individuals with origins in any of the black racial groups of africa, including, for example, african american, jamaican, haitian, nigerian, ethiopian, and somali.",
        "E3": "hispanic or latino - includes individuals of mexican, puerto rican, salvadoran, cuban, dominican, guatemalan, and other central or south american or spanish culture or origin.",
        "E4": "asian - individuals with origins in any of the original peoples of central or east asia, southeast asia, or south asia, including, for example, chinese, asian indian, filipino, vietnamese, korean, and japanese.",
        "E5": "american indian or alaskan native or native hawaiian or pacific islander - individuals with origins in any of the original peoples of north, central, and south america, including, for example, navajo nation, blackfeet tribe of the blackfeet the indian reservation of montana, native village of barrow inupiat traditional government, nome eskimo community, aztec, and maya -- as well as individuals with origins in any of the original peoples of hawaii, guam, samoa, or other pacific islands, including, for example, native hawaiian, samoan, chamorro, tongan, fijian, and marshallese.",
        "E6": "multiracial - individuals who identify explicitly as belonging to more than one of the racial and ethnic groups above, such as biracial individuals with one white and one black parent, or those with a combination of asian and hispanic heritage, etc. mixed-race individuals often face unique social experiences, such as celebrating diverse cultural holidays, speaking multiple languages, and bridging different cultural perspectives within their families and communities."
      }
    },
    {
      "title": "AGE",
      "options": {
        "A1": "under 18 years old",
        "A2": "18 to 24 years old",
        "A3": "25 to 34 years old",
        "A4": "35 to 44 years old",
        "A5": "45 to 54 years old",
        "A6": "55 to 64 years old",
        "A7": "65 or older"
      }
    }
]

# ------------------------------------------------------------
# 1. Endpoint & authentication
# ------------------------------------------------------------
endpoint_base = "https://aipolling.services.ai.azure.com/models"
api_version = "2024-05-01-preview"
url = f"{endpoint_base}/chat/completions?api-version={api_version}"

# Load API key from environment variable
api_key = os.getenv("AZURE_API_KEY")
if not api_key:
    raise ValueError("AZURE_API_KEY environment variable is not set")

# ------------------------------------------------------------
# 3. Base prompt
# ------------------------------------------------------------
prompt_template = """
A social media account has the following username, name, description:
- Name: {name}  
- Username: {username}  
- Description: {description}  
A structured description of their profile image is the following:
- Profile Image Description: {profile_image_description}
Furthermore, they self-report their location in their bio as: {location}  
The amount of followers they have is: {followers}  
They have written the following tweet(s): {tweets}  

I will show you a number of categories to which this user may belong to.
The categories are preceded by a title (e.g. "AGE:" or "SEX:" etc.) and a symbol (e.g. "A1", "A2" or "E1" etc.).
Please select, for each title, the most likely category to which this user belongs to.

In your answer present, for each title, the selected symbol.
Write out in full the category associated with the selected symbol.
The chosen symbol / category must be the most likely to accurately represent this user.
You must only select one symbol / category per title.
A title, symbol and category cannot appear more than once in your answer.

For each selected symbol / category, please note the level of Speculation involved in this selection.
Present the Speculation level for each selection on a scale from 0 (not speculative at all, every single element of the user data was useful in the selection) to 100 (fully speculative, there is no information related to this title in the user data).
Speculation levels should be a direct measure of the amount of useful information available in the user data.
Speculation levels pertain only to the information available in the user data -- namely the username, name, description, location, profile picture and tweets from this user -- and should not be affected by additional information available to you from any other source.

To ensure consistency, use the following guidelines to determine speculation levels:

0-20 (Low speculation): The user data provides clear and direct information relevant to the title. (e.g., explicit mention in the profile or tweets)
21-40 (Moderate-low speculation): The user data provides indirect but strong indicators relevant to the title.
41-60 (Moderate speculation): The user data provides some hints or partial information relevant to the title.
61-80 (Moderate-high speculation): The user data provides limited and weak indicators relevant to the title.
81-100 (High speculation): The user data provides no or almost no information relevant to the title.

For each selected category, please explain at length what features of the data contributed to your choice and your speculation level.

Preserve a strictly structured answer to ease parsing of the text.
Format your output as follows (this is just an example):

title: AGE
explanation: ...
symbol: A1)
category: 18-25
speculation: 90

For every title, you must include exactly these 5 fields: title, explanation, symbol, category, speculation. Do not omit any of them.
Please keep the EXACT WORDING for the titles!
YOU MUST GIVE AN ANSWER FOR EVERY TITLE !
    """

# ------------------------------------------------------------
# 4. Build dynamic belief prompt
# ------------------------------------------------------------
def build_belief_prompt(user_row, beliefs):
    
    #randomly shuffle the belives to prevent position bias
    shuffled_beliefs = beliefs[:]
    random.shuffle(shuffled_beliefs)

    categories_text = "\n".join(
        [
            f"- **{belief['title']}:** \n    Choose one: [\n    " +
            ", \n    ".join([f"{key}) {val}" for key, val in belief["options"].items()]) +
            "]"
            for belief in shuffled_beliefs
        ]
    )

    return prompt_template.format(
        name=user_row['name'],
        username=user_row['username'],
        description=user_row['description'],
        location=user_row['location'],
        followers=user_row['public_metrics.followers_count'],
        tweets=user_row['tweets_struct'],
        profile_image_description=user_row['profile_image_main_info'],
    ) + "\n\nBelow is the list of categories to which this user may belong to:\n" + categories_text

# ----------------------------
# 5. Result Parsing Function
# ----------------------------

def parse_belief_response(response, beliefs, prefix=None):
    categories = {}
    explanations = {}
    speculations = {}
    symbols = {}

    for belief in beliefs:
        safe_key = re.sub(r"\W+", "_", belief["title"])
        if prefix:
            safe_key = f"{prefix}_{safe_key}"
        categories[f"{safe_key}_Category"] = None
        explanations[f"{safe_key}_Explanation"] = None
        symbols[f"{safe_key}_Symbol"] = None
        speculations[f"{safe_key}_Speculation"] = None

    belief_sections = re.split(r"(?=title:\s*)", str(response), flags=re.IGNORECASE)
    belief_sections = [section.strip() for section in belief_sections if section.strip() and not section.lower().startswith('below')]
    print(f"[DEBUG] Found {len(belief_sections)} sections to parse") 
    
    # Pattern for individual belief
    pattern = (
        r"title:?\s*(.*?)\s*[\r\n]"
        r"explanation:?\s*(.*?)\s*[\r\n]"
        r"symbol:?\s*(.*?)\s*[\r\n]"
        r"category:?\s*(.*?)\s*[\r\n]"
        r"speculation:?\s*(\d+)\s*"
    )
    
    for i, section in enumerate(belief_sections):
        
        match = re.search(pattern, section, re.DOTALL | re.IGNORECASE)
        
        if match:
            title = normalize_text(match.group(1))
            explanation = normalize_text(match.group(2))
            symbol = normalize_text(match.group(3))
            category = normalize_text(match.group(4))
            speculation = normalize_text(match.group(5))
            
            # Find best matching belief title from JSON
            matched_belief = None
            for b in beliefs:
                if b["title"] in title or title in b["title"]:
                    matched_belief = b
                    break
            if matched_belief:
                safe_key = re.sub(r"\W+", "_", matched_belief["title"])
                if prefix:
                    safe_key = f"{prefix}_{safe_key}"
                categories[f"{safe_key}_Category"] = category.strip()
                explanations[f"{safe_key}_Explanation"] = explanation.strip()
                symbols[f"{safe_key}_Symbol"] = symbol.strip()
                speculations[f"{safe_key}_Speculation"] = int(speculation.strip())

    return {**categories, **explanations, **speculations, **symbols}


# ------------------------------------------------------------
# 6. Build the request body
# ------------------------------------------------------------
def infer_beliefs(row, model_name, features):

    prompt = build_belief_prompt(row, features)

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    body = {
        "model": model_name,
        "max_tokens": 16384,
        "messages": messages
    }

    headers = {
        "Content-Type": "application/json",
        "api-key": api_key
    }

    response = requests.post(url, headers=headers, data=json.dumps(body))

    response.raise_for_status()

    content_json = response.json()
    reply = content_json["choices"][0]["message"]["content"]
    return reply

# ------------------------------------------------------------
# 7. Secondary GPT cleanup function
# ------------------------------------------------------------
cleanup_prompt_template = """
You will be shown a long-form response to a question asking to assign a series of categories to a user.
Your task is to extract and reformat ONLY the belief category assessments into a strict format.

CRITICAL FORMATTING RULES:
- Each belief must be formatted EXACTLY as follows:
title: <FULL TITLE TEXT>
explanation: <FULL EXPLANATION TEXT>
symbol: <SYMBOL CODE>
category: <FULL CATEGORY TEXT>
speculation: <NUMERIC VALUE ONLY>

- Each field must be on its own separate line
- Preserve ALL original content from explanations, categories, and titles - do not summarize or shorten
- For speculation, use ONLY the numeric value (e.g., "50" not "fifty" or "50%")
- Remove any asterisks, markdown, or extra formatting from the original
- Ensure there is exactly one blank line between each complete belief assessment
- If any field is missing, include the field name with nothing after it (e.g., "explanation:")

CHARACTER AND ENCODING RULES:
- Use only standard ASCII hyphens (-) for all dashes and hyphens in titles and categories
- Replace any special dashes (‑, –, —) with regular hyphens (-)
- Ensure all text uses standard ASCII characters only
- Remove any non-ASCII characters or convert them to their closest ASCII equivalent
- Use regular parentheses () not special parentheses

EXAMPLES OF CORRECT FORMATTING:

Example 1:
title: OPINION ON WHETHER DIVORCE SHOULD BE EASIER OR MORE DIFFICULT
explanation: The user's tweet mentions divorce in a positive context...
symbol: SoA1)
category: Obtaining a divorce should be easier than it is
speculation: 50

Example 2:
title: POLITICAL AFFILIATION
explanation: The user frequently praises conservative figures...
symbol: PoA3)
category: Republican
speculation: 75

COMMON MISTAKES TO AVOID:
- WRONG: speculation: fifty (50)
- CORRECT: speculation: 50

- WRONG: symbol: SoA1) (with extra spaces)
- CORRECT: symbol: SoA1)

- WRONG: Using markdown like **title:**
- CORRECT: title:

- WRONG: Combining multiple fields on one line
- CORRECT: Each field on its own line

- WRONG: title: OPINION ON ISRAEL‑HAMAS WAR (special hyphen)
- CORRECT: title: OPINION ON ISRAEL-HAMAS WAR (regular hyphen)

- WRONG: category: Obtaining a divorce should be easier than it is (special characters)
- CORRECT: category: Obtaining a divorce should be easier than it is

Now, clean and reformat the following response to match this exact structure. Preserve all original meaning but enforce the strict formatting and character rules:

{raw_output}
"""

def clean_gpt_output(raw_output, model_name):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": cleanup_prompt_template.format(raw_output=raw_output)}
    ]

    body = {
        "model": model_name,
        "max_tokens": 16384,
        "messages": messages
    }

    headers = {
        "Content-Type": "application/json",
        "api-key": api_key
    }

    response = requests.post(url, headers=headers, data=json.dumps(body))
    response.raise_for_status()
    content_json = response.json()
    reply = content_json["choices"][0]["message"]["content"]
    return reply

# ------------------------------------------------------------
# 8. GPT call with retry logic
# ------------------------------------------------------------
def infer_beliefs_with_retry(row, model_name, features, prefix=None, max_retries=15):
    for attempt in range(1, max_retries + 1):
        response_text = None
        cleaned_text = None
        try:
            response_text = infer_beliefs(row, model_name, features)
            parsed_result = parse_belief_response(response_text, features, prefix=prefix)

            # if parsed result is incomplete, proceed with seecondary cleanup
            if any(value is None for value in parsed_result.values()):
                print(f"[DEBUG] Parsing incomplete for {row['username']} → attempting cleanup.")
                cleaned_text = clean_gpt_output(response_text, model_name)
                parsed_result = parse_belief_response(cleaned_text, features, prefix=prefix)

            # if still incomplete, raise exception to retry full GPT
            if any(value is None for value in parsed_result.values()):
                raise ValueError("Parsed result is empty. Retrying...")
            else:
                print(f"Parsing succeeded directly for {row['username']}")
            return response_text, cleaned_text, parsed_result

        except Exception as e:
            print(f"Attempt {attempt}/{max_retries} failed for user {row['username']} ({model_name}): {e}")
            if attempt < max_retries:
                time.sleep(2 ** attempt * 0.1)  
    return response_text, cleaned_text, parsed_result

# ------------------------------------------------------------
# 9. Apply inference to the dataframe
# ------------------------------------------------------------
models = [
    "gpt-oss-120b"
]

df = merged_df
all_results = []
user_results = defaultdict(list)

output_file = "../generated_data/feature_extraction/all_results.csv"
user_dir = "../generated_data/feature_extraction/users"
os.makedirs(user_dir, exist_ok=True)

# load results from checkpoint
if os.path.exists(output_file):
    all_results = pd.read_csv(output_file, encoding="utf-8-sig").to_dict("records")

checkpoint = load_checkpoint()
start_index = checkpoint["last_index"] + 1
print(f"Resuming: {start_index} rows already processed.")

completed_users = set()
total_users = df['id'].nunique()

for model_name in models:
    current_user = None
    
    for index, row in df.iloc[start_index:].iterrows():
        user_id = row["id"]

        if current_user != user_id:
            if current_user is not None:
                # Save completed previous user
                pd.DataFrame(user_results[current_user]).to_csv(
                    f"{user_dir}/{current_user}.csv", index=False, encoding="utf-8-sig"
                )
                print(f"Finished all timewindows for user {current_user}")
                completed_users.add(current_user)
                print(f"{len(completed_users)}/{total_users} users completed")

            # Switch to new user
            current_user = user_id

            # Load existing user CSV if exists to resume
            user_file = f"{user_dir}/{user_id}.csv"
            if os.path.exists(user_file):
                user_results[user_id] = pd.read_csv(user_file, encoding="utf-8-sig").to_dict("records")
            else:
                user_results[user_id] = []
        try:
            combined_row = row.to_dict()
            # with empty windows, set all belief/demo outputs to None
            if row.get("is_empty_window", False):
                combined_row = row.to_dict()
                combined_row.update({
                    "Model": model_name,
                    "Belief_Raw": None,
                    "Belief_Cleaned": None,
                    "Demo_Raw": None,
                    "Demo_Cleaned": None
                })

                for feature in belief_features:
                    safe_key = re.sub(r"\W+", "_", feature["title"])
                    combined_row.update({
                        f"Belief_{safe_key}_Category": None,
                        f"Belief_{safe_key}_Explanation": None,
                        f"Belief_{safe_key}_Symbol": None,
                        f"Belief_{safe_key}_Speculation": None
                    })
                for feature in demo_features:
                    safe_key = re.sub(r"\W+", "_", feature["title"])
                    combined_row.update({
                        f"Demo_{safe_key}_Category": None,
                        f"Demo_{safe_key}_Explanation": None,
                        f"Demo_{safe_key}_Symbol": None,
                        f"Demo_{safe_key}_Speculation": None
                    })

                all_results.append(combined_row)
                user_results[user_id].append(combined_row)
                save_checkpoint(index)
                continue 
            # Belief features
            belief_text, cleaned_belief, parsed_belief = infer_beliefs_with_retry(row, model_name, belief_features, prefix="Belief")

            # Demo features
            demo_text, cleaned_demo, parsed_demo = infer_beliefs_with_retry(row, model_name, demo_features, prefix="Demo")

            combined_row.update({
                "Model": model_name,
                "Belief_Raw": belief_text,
                "Belief_Cleaned": cleaned_belief,
                "Demo_Raw": demo_text,
                "Demo_Cleaned": cleaned_demo
            })
            combined_row.update(parsed_belief)
            combined_row.update(parsed_demo)
            all_results.append(combined_row)
            user_results[user_id].append(combined_row)

            pd.DataFrame(all_results).to_csv("../generated_data/feature_extraction/all_results.csv", index=False, encoding="utf-8-sig")

            pd.DataFrame(user_results[user_id]).to_csv(
                f"{user_dir}/{user_id}.csv", index=False, encoding="utf-8-sig"
            )

            if (index + 1) % 50 == 0:
                print(f"Progess: {index + 1}/{len(df)} rows completed")
            save_checkpoint(index)
            time.sleep(1)  
        except Exception as e:
            print(f"Error processing row {index} for model {model_name}: {e}")

# for the last user
if current_user is not None and current_user not in completed_users:
    pd.DataFrame(user_results[current_user]).to_csv(
        f"{user_dir}/{current_user}.csv", index=False, encoding="utf-8-sig"
    )
    print(f"Finished all timewindows for user {current_user}")
    completed_users.add(current_user)
    print(f"{len(completed_users)}/{total_users} users completed")

print("Done!")


In [ ]:
import requests
import json
import re
import unicodedata
import random

# ------------------------------
# Minimal normalize function
# ------------------------------
def normalize_text(text):
    if not text:
        return text
    text = str(text)
    text = text.replace('*', '').replace('<<', '').replace('>>', '')
    text = re.sub(r'[-\u2010-\u2015\u2212\uFE58\uFE63\uFF0D]', '', text)
    normalized = unicodedata.normalize('NFKD', text)
    ascii_text = normalized.encode('ASCII', 'ignore').decode('ASCII')
    return ' '.join(ascii_text.split()).lower()

# ------------------------------
# Minimal example user input
# ------------------------------
user_row = {
    "id": 1,
    "name": "John Doe",
    "username": "johndoe123",
    "description": "Tech enthusiast and blogger.",
    "location": "New York, USA",
    "public_metrics.followers_count": 250,
    "tweets_struct": "Just posted a new blog about AI!",
    "profile_image_main_info": "A smiling man with glasses"
}

# ------------------------------
# Minimal example belief features
# ------------------------------
belief_features = [
    {
        "title": "AGE",
        "options": {"A1": "18-25", "A2": "26-35", "A3": "36-45"}
    },
    {
        "title": "SEX",
        "options": {"S1": "Male", "S2": "Female"}
    }
]

# ------------------------------
# Prompt builder
# ------------------------------
prompt_template = """
A social media account has the following username, name, description:
- Name: {name}  
- Username: {username}  
- Description: {description}  
A structured description of their profile image is the following:
- Profile Image Description: {profile_image_description}
Furthermore, they self-report their location in their bio as: {location}  
The amount of followers they have is: {followers}  
They have written the following tweet(s): {tweets}  

I will show you a number of categories to which this user may belong to.
The categories are preceded by a title (e.g. "AGE:" or "SEX:" etc.) and a symbol (e.g. "A1", "A2" or "E1" etc.).
Please select, for each title, the most likely category to which this user belongs to.

In your answer present, for each title, the selected symbol.
Write out in full the category associated with the selected symbol.
The chosen symbol / category must be the most likely to accurately represent this user.
You must only select one symbol / category per title.
A title, symbol and category cannot appear more than once in your answer.

For each selected symbol / category, please note the level of Speculation involved in this selection.
Present the Speculation level for each selection on a scale from 0 (not speculative at all, every single element of the user data was useful in the selection) to 100 (fully speculative, there is no information related to this title in the user data).
Speculation levels should be a direct measure of the amount of useful information available in the user data.
Speculation levels pertain only to the information available in the user data -- namely the username, name, description, location, profile picture and tweets from this user -- and should not be affected by additional information available to you from any other source.

To ensure consistency, use the following guidelines to determine speculation levels:

0-20 (Low speculation): The user data provides clear and direct information relevant to the title. (e.g., explicit mention in the profile or tweets)
21-40 (Moderate-low speculation): The user data provides indirect but strong indicators relevant to the title.
41-60 (Moderate speculation): The user data provides some hints or partial information relevant to the title.
61-80 (Moderate-high speculation): The user data provides limited and weak indicators relevant to the title.
81-100 (High speculation): The user data provides no or almost no information relevant to the title.

For each selected category, please explain at length what features of the data contributed to your choice and your speculation level.

Preserve a strictly structured answer to ease parsing of the text.
Format your output as follows (this is just an example):

title: AGE
explanation: ...
symbol: A1)
category: 18-25
speculation: 90

For every title, you must include exactly these 5 fields: title, explanation, symbol, category, speculation. Do not omit any of them.
Please keep the EXACT WORDING for the titles!
YOU MUST GIVE AN ANSWER FOR EVERY TITLE !
"""

def build_belief_prompt(user_row, beliefs):
    categories_text = "\n".join(
        [
            f"- {belief['title']}: " +
            ", ".join([f"{key}) {val}" for key, val in belief["options"].items()])
            for belief in beliefs
        ]
    )
    return prompt_template.format(
        name=user_row['name'],
        username=user_row['username'],
        description=user_row['description'],
        location=user_row['location'],
        followers=user_row['public_metrics.followers_count'],
        tweets=user_row['tweets_struct'],
        profile_image_description=user_row['profile_image_main_info'],
    ) + "\n" + categories_text

# ------------------------------
# API request
# ------------------------------
endpoint_base = "https://aipolling.services.ai.azure.com/models"
api_version = "2024-05-01-preview"
url = f"{endpoint_base}/chat/completions?api-version={api_version}"
api_key = '3NmIwZ7KyeIwcErih36euZvzo7aOWb956ailqXO3sOCrWO6W5zEzJQQJ99BEAC5RqLJXJ3w3AAAAACOG244L' 

def infer_beliefs(user_row, model_name, features):
    prompt = build_belief_prompt(user_row, features)
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    body = {
        "model": model_name,
        "max_tokens": 500,
        "messages": messages
    }
    headers = {"Content-Type": "application/json", "api-key": api_key}
    response = requests.post(url, headers=headers, data=json.dumps(body))
    response.raise_for_status()
    content_json = response.json()
    reply = content_json["choices"][0]["message"]["content"]
    return reply

# ------------------------------
# Run a single reproducible query
# ------------------------------
model_name = "gpt-oss-120b"
response = infer_beliefs(user_row, model_name, belief_features)
print("Response:\n", response)


Response:
 title: AGE  
explanation: The profile gives no direct indication of the user’s age. The only clues are indirect: the user calls themselves a “tech enthusiast and blogger,” has a modest follower count (250), and is posting about a new AI blog. This type of activity is most commonly associated with adults who have completed at least some higher‑education or early‑career experience, which generally falls into the late‑twenties to early‑thirties range. No explicit age‑related words appear in the name, username, location, image description, or tweet, so the inference is based on typical demographic patterns for tech‑focused micro‑blog
